# Day 2 — Manual Autoregressive Generation

**50-Day ML Infrastructure & LLM Systems Roadmap**

This notebook implements autoregressive text generation manually with DistilGPT-2, without using `model.generate()` for the main implementation.

Topics covered:

- tokenization
- logits
- softmax
- greedy decoding
- multinomial sampling
- temperature scaling
- autoregressive generation
- latency
- throughput
- per-token decoding cost


## 1. Environment Setup

Install the required libraries and inspect the Colab runtime.


In [ ]:
!pip install -q transformers accelerate psutil pandas matplotlib


In [ ]:
import platform
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

runtime_memory = psutil.virtual_memory()
total_runtime_ram_gb = runtime_memory.total / 1024**3
available_ram_before_loading_gb = runtime_memory.available / 1024**3

environment_info = {
    "device": str(device).upper(),
    "cuda_device": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    ),
    "python_version": platform.python_version(),
    "pytorch_version": torch.__version__,
    "total_runtime_ram_gb": total_runtime_ram_gb,
    "available_ram_before_loading_gb": available_ram_before_loading_gb,
}

for key, value in environment_info.items():
    print(f"{key}: {value}")


## 2. Load DistilGPT-2

The model is loaded in evaluation mode because this notebook performs inference only.


In [ ]:
MODEL_NAME = "distilbert/distilgpt2"

process = psutil.Process()
process_ram_before_loading_mb = process.memory_info().rss / 1024**2

load_start = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

model.to(device)
model.eval()

load_time_seconds = time.perf_counter() - load_start

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

parameter_count = sum(parameter.numel() for parameter in model.parameters())
estimated_parameter_memory_mb = parameter_count * 4 / 1024**2

process_ram_after_loading_mb = process.memory_info().rss / 1024**2
process_ram_increase_mb = (
    process_ram_after_loading_mb - process_ram_before_loading_mb
)

model_info = {
    "model_name": MODEL_NAME,
    "parameter_count": parameter_count,
    "estimated_parameter_memory_mb": estimated_parameter_memory_mb,
    "load_time_seconds": load_time_seconds,
    "process_ram_before_loading_mb": process_ram_before_loading_mb,
    "process_ram_after_loading_mb": process_ram_after_loading_mb,
    "process_ram_increase_mb": process_ram_increase_mb,
}

for key, value in model_info.items():
    print(f"{key}: {value}")


## 3. Inspect Tokenization

The tokenizer converts text into token IDs. The model predicts the next token from this sequence.


In [ ]:
PROMPT = "Artificial intelligence infrastructure is important because"
MAX_NEW_TOKENS = 25
WARMUP_RUNS = 2
MEASURED_RUNS = 5
TEMPERATURES = [0.2, 0.7, 1.0, 1.5, 2.0]

encoded_prompt = tokenizer(PROMPT, return_tensors="pt")
input_ids = encoded_prompt["input_ids"].to(device)
input_token_count = input_ids.shape[1]

print(f"Prompt: {PROMPT}")
print(f"Input token count: {input_token_count}")
print(f"Input IDs: {input_ids}")
print(f"Tokens: {tokenizer.convert_ids_to_tokens(input_ids[0])}")


## 4. Inspect Logits and Softmax

The model outputs logits. Softmax converts the final-position logits into next-token probabilities.


In [ ]:
with torch.inference_mode():
    outputs = model(input_ids=input_ids)

all_logits = outputs.logits
next_token_logits = all_logits[:, -1, :]
next_token_probabilities = torch.softmax(next_token_logits, dim=-1)

print(f"All logits shape: {tuple(all_logits.shape)}")
print(f"Next-token logits shape: {tuple(next_token_logits.shape)}")
print(f"Probability sum: {next_token_probabilities.sum().item():.6f}")


In [ ]:
top_probabilities, top_token_ids = torch.topk(
    next_token_probabilities,
    k=10,
    dim=-1,
)

top_token_rows = []

for rank, (probability, token_id) in enumerate(
    zip(top_probabilities[0], top_token_ids[0]),
    start=1,
):
    top_token_rows.append(
        {
            "rank": rank,
            "token_id": token_id.item(),
            "token": repr(tokenizer.decode([token_id.item()])),
            "probability": probability.item(),
        }
    )

top_tokens_df = pd.DataFrame(top_token_rows)
top_tokens_df


## 5. Manual Greedy Decoding

Greedy decoding selects the highest-scoring token at each step.


In [ ]:
def synchronize_device() -> None:
    if device.type == "cuda":
        torch.cuda.synchronize()


def manual_greedy_generate(
    prompt: str,
    max_new_tokens: int,
    stop_on_eos: bool = True,
) -> dict:
    encoded = tokenizer(prompt, return_tensors="pt")
    generated_ids = encoded["input_ids"].to(device)

    generated_token_ids = []
    step_latencies = []

    with torch.inference_mode():
        for _ in range(max_new_tokens):
            synchronize_device()
            step_start = time.perf_counter()

            outputs = model(input_ids=generated_ids)
            next_token_logits = outputs.logits[:, -1, :]
            next_token_id = torch.argmax(
                next_token_logits,
                dim=-1,
                keepdim=True,
            )

            generated_ids = torch.cat(
                [generated_ids, next_token_id],
                dim=-1,
            )

            synchronize_device()
            step_latency = time.perf_counter() - step_start

            step_latencies.append(step_latency)
            generated_token_ids.append(next_token_id.item())

            if (
                stop_on_eos
                and next_token_id.item() == tokenizer.eos_token_id
            ):
                break

    generated_text = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True,
    )

    total_time = sum(step_latencies)
    generated_token_count = len(generated_token_ids)

    return {
        "text": generated_text,
        "generated_token_ids": generated_token_ids,
        "generated_tokens": generated_token_count,
        "step_latencies": step_latencies,
        "total_time": total_time,
        "tokens_per_second": (
            generated_token_count / total_time
            if total_time > 0
            else 0.0
        ),
    }


In [ ]:
greedy_example = manual_greedy_generate(
    prompt=PROMPT,
    max_new_tokens=MAX_NEW_TOKENS,
)

print(greedy_example["text"])
print()
print(f"Generated tokens: {greedy_example['generated_tokens']}")
print(f"Total time: {greedy_example['total_time']:.4f} seconds")
print(
    f"Throughput: {greedy_example['tokens_per_second']:.4f} tokens/second"
)


## 6. Manual Sampling with Temperature

Sampling draws the next token from the probability distribution.

Temperature controls the sharpness of that distribution.


In [ ]:
def manual_sample_generate(
    prompt: str,
    max_new_tokens: int,
    temperature: float,
    seed: int | None = None,
    stop_on_eos: bool = True,
) -> dict:
    if temperature <= 0:
        raise ValueError("temperature must be greater than zero")

    if seed is not None:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)

    encoded = tokenizer(prompt, return_tensors="pt")
    generated_ids = encoded["input_ids"].to(device)

    generated_token_ids = []
    step_latencies = []

    with torch.inference_mode():
        for _ in range(max_new_tokens):
            synchronize_device()
            step_start = time.perf_counter()

            outputs = model(input_ids=generated_ids)
            next_token_logits = outputs.logits[:, -1, :]
            scaled_logits = next_token_logits / temperature
            probabilities = torch.softmax(scaled_logits, dim=-1)

            next_token_id = torch.multinomial(
                probabilities,
                num_samples=1,
            )

            generated_ids = torch.cat(
                [generated_ids, next_token_id],
                dim=-1,
            )

            synchronize_device()
            step_latency = time.perf_counter() - step_start

            step_latencies.append(step_latency)
            generated_token_ids.append(next_token_id.item())

            if (
                stop_on_eos
                and next_token_id.item() == tokenizer.eos_token_id
            ):
                break

    generated_text = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True,
    )

    total_time = sum(step_latencies)
    generated_token_count = len(generated_token_ids)

    return {
        "text": generated_text,
        "generated_token_ids": generated_token_ids,
        "generated_tokens": generated_token_count,
        "step_latencies": step_latencies,
        "total_time": total_time,
        "tokens_per_second": (
            generated_token_count / total_time
            if total_time > 0
            else 0.0
        ),
        "temperature": temperature,
    }


In [ ]:
sampling_examples = {}

for temperature in TEMPERATURES:
    result = manual_sample_generate(
        prompt=PROMPT,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=temperature,
        seed=SEED,
    )
    sampling_examples[temperature] = result

    print(f"Temperature: {temperature}")
    print(result["text"])
    print("-" * 100)


## 7. Measure the Effect of Temperature

Entropy indicates how spread out the probability distribution is.

Higher entropy generally means a less concentrated distribution.


In [ ]:
temperature_rows = []

with torch.inference_mode():
    base_logits = model(input_ids=input_ids).logits[:, -1, :]

for temperature in TEMPERATURES:
    probabilities = torch.softmax(
        base_logits / temperature,
        dim=-1,
    )

    top_probability, top_token_id = torch.max(
        probabilities,
        dim=-1,
    )

    entropy = -(
        probabilities * torch.log(probabilities + 1e-12)
    ).sum(dim=-1)

    temperature_rows.append(
        {
            "temperature": temperature,
            "top_token": repr(
                tokenizer.decode([top_token_id.item()])
            ),
            "top_probability": top_probability.item(),
            "entropy": entropy.item(),
        }
    )

temperature_df = pd.DataFrame(temperature_rows)
temperature_df


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    temperature_df["temperature"],
    temperature_df["top_probability"],
    marker="o",
)
plt.xlabel("Temperature")
plt.ylabel("Top-token probability")
plt.title("Temperature vs Top-token Probability")
plt.grid(True)
plt.show()


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    temperature_df["temperature"],
    temperature_df["entropy"],
    marker="o",
)
plt.xlabel("Temperature")
plt.ylabel("Entropy")
plt.title("Temperature vs Distribution Entropy")
plt.grid(True)
plt.show()


## 8. Benchmark Greedy and Sampling Strategies

Each configuration uses warm-up runs followed by repeated measured runs.


In [ ]:
def benchmark_generation(
    method: str,
    temperature: float | None = None,
) -> dict:
    if method not in {"greedy", "sampling"}:
        raise ValueError("method must be 'greedy' or 'sampling'")

    def run_once(run_seed: int) -> dict:
        if method == "greedy":
            return manual_greedy_generate(
                prompt=PROMPT,
                max_new_tokens=MAX_NEW_TOKENS,
            )

        return manual_sample_generate(
            prompt=PROMPT,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=temperature,
            seed=run_seed,
        )

    for warmup_index in range(WARMUP_RUNS):
        run_once(SEED + warmup_index)

    latencies = []
    token_counts = []
    outputs = []
    all_step_latencies = []

    for run_index in range(MEASURED_RUNS):
        synchronize_device()
        benchmark_start = time.perf_counter()

        result = run_once(SEED + 100 + run_index)

        synchronize_device()
        elapsed = time.perf_counter() - benchmark_start

        latencies.append(elapsed)
        token_counts.append(result["generated_tokens"])
        outputs.append(result["text"])
        all_step_latencies.extend(result["step_latencies"])

    mean_latency = float(np.mean(latencies))
    mean_generated_tokens = float(np.mean(token_counts))

    return {
        "method": method,
        "temperature": (
            temperature if method == "sampling" else None
        ),
        "mean_latency_seconds": mean_latency,
        "median_latency_seconds": float(np.median(latencies)),
        "minimum_latency_seconds": float(np.min(latencies)),
        "maximum_latency_seconds": float(np.max(latencies)),
        "latency_std_seconds": float(np.std(latencies)),
        "mean_generated_tokens": mean_generated_tokens,
        "mean_step_latency_seconds": float(
            np.mean(all_step_latencies)
        ),
        "tokens_per_second": (
            mean_generated_tokens / mean_latency
            if mean_latency > 0
            else 0.0
        ),
        "sample_output": outputs[0],
    }


In [ ]:
benchmark_results = [
    benchmark_generation(method="greedy")
]

for temperature in TEMPERATURES:
    benchmark_results.append(
        benchmark_generation(
            method="sampling",
            temperature=temperature,
        )
    )

benchmark_df = pd.DataFrame(benchmark_results)
benchmark_df


In [ ]:
benchmark_display_columns = [
    "method",
    "temperature",
    "mean_latency_seconds",
    "median_latency_seconds",
    "minimum_latency_seconds",
    "maximum_latency_seconds",
    "latency_std_seconds",
    "mean_generated_tokens",
    "mean_step_latency_seconds",
    "tokens_per_second",
]

benchmark_df[benchmark_display_columns].round(4)


## 9. Plot Per-Token Latency

This plot shows whether later generation steps become slower as the sequence grows.


In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(
    range(1, len(greedy_example["step_latencies"]) + 1),
    greedy_example["step_latencies"],
    marker="o",
)
plt.xlabel("Generated token step")
plt.ylabel("Step latency in seconds")
plt.title("Greedy Decoding Latency per Token")
plt.grid(True)
plt.show()


## 10. Validate Manual Greedy Decoding

`model.generate()` is used only here to verify the manual implementation.


In [ ]:
hugging_face_inputs = tokenizer(
    PROMPT,
    return_tensors="pt",
).to(device)

with torch.inference_mode():
    hugging_face_output_ids = model.generate(
        **hugging_face_inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

hugging_face_text = tokenizer.decode(
    hugging_face_output_ids[0],
    skip_special_tokens=True,
)

outputs_match = greedy_example["text"] == hugging_face_text

print(f"Outputs match: {outputs_match}")
print()
print("Manual greedy output:")
print(greedy_example["text"])
print()
print("Hugging Face greedy output:")
print(hugging_face_text)
